# 04 — XLM-R

Chạy **XLM-R** cho cả hai tác vụ.

### Training protocol
- max epochs = **10**
- early stopping patience = **3**
- checkpoint metric = **Dev Macro-F1**
- optimizer = AdamW
- LR = `2e-5`
- weight decay = `0.01`
- warmup ratio = `0.10`
- scheduler = cosine
- gradient clipping = `1.0`
- effective batch ≈ `32`
- mixed precision FP16
- weighted Cross Entropy
- log thêm PR-AUC của positive class
- Test chỉ sau khi restore best Dev checkpoint

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip -q install   "transformers==4.49.0"   "accelerate==1.4.0"   "datasets==3.3.2"   "scikit-learn==1.6.1"   "sentencepiece==0.2.0"   "pandas==2.2.2"   "tqdm==4.67.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==

In [3]:
import os, gc, json, math, time, random, shutil, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    average_precision_score,
)
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Hãy bật GPU."

ROOT = Path("/content/drive/MyDrive/DS108_Toxic_4Models")
SPLIT_DIR = ROOT / "data_splits"
ARTIFACT_DIR = ROOT / "artifacts"
RESULT_DIR = ROOT / "results"
PRED_DIR = RESULT_DIR / "predictions"
CM_DIR = RESULT_DIR / "confusion_matrices"
CACHE_DIR = ROOT / "cache"

for p in [ARTIFACT_DIR, RESULT_DIR, PRED_DIR, CM_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert (SPLIT_DIR / "train.csv").exists(), "Chạy 00_Prepare_Data.ipynb trước."

MAX_EPOCHS = 10
PATIENCE = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRAD_CLIP = 1.0

TASKS = {
    "task1": {
        "label_col": "toxicity",
        "label2id": {"NON-TOXIC": 0, "TOXIC": 1},
        "id2label": {0: "NON-TOXIC", 1: "TOXIC"},
    },
    "task2": {
        "label_col": "expression_type",
        "label2id": {"EXPLICIT": 0, "IMPLICIT": 1},
        "id2label": {0: "EXPLICIT", 1: "IMPLICIT"},
    },
}

def seed_all():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    os.environ["PYTHONHASHSEED"] = str(SEED)

def get_task(split_name, task):
    x = pd.read_csv(SPLIT_DIR / f"{split_name}.csv")
    if task == "task2":
        x = x[x["toxicity"].eq("TOXIC")].copy()
    x["label_id"] = x[TASKS[task]["label_col"]].map(TASKS[task]["label2id"])
    assert x["label_id"].notna().all()
    return x.reset_index(drop=True)

def class_weights(train_frame):
    c = train_frame["label_id"].value_counts().sort_index()
    n, k = len(train_frame), len(c)
    return torch.tensor([n / (k * c.loc[i]) for i in range(k)], dtype=torch.float32)

def metrics(y, yp, probs):
    acc = accuracy_score(y, yp)
    mp, mr, mf, _ = precision_recall_fscore_support(
        y, yp, average="macro", zero_division=0
    )
    _, _, wf, _ = precision_recall_fscore_support(
        y, yp, average="weighted", zero_division=0
    )
    pr_auc = average_precision_score(y, probs[:, 1])
    return {
        "accuracy": float(acc),
        "macro_precision": float(mp),
        "macro_recall": float(mr),
        "macro_f1": float(mf),
        "weighted_f1": float(wf),
        "pr_auc_positive_class": float(pr_auc),
    }

def save_json(obj, path):
    Path(path).write_text(
        json.dumps(obj, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

In [4]:
MODEL_KEY = "xlmr"
MODEL_ID = "FacebookAI/xlm-roberta-base"

print("GPU:", torch.cuda.get_device_name(0))
print("MODEL:", MODEL_ID)
print("MAX_EPOCHS:", MAX_EPOCHS, "| PATIENCE:", PATIENCE)

GPU: Tesla T4
MODEL: FacebookAI/xlm-roberta-base
MAX_EPOCHS: 10 | PATIENCE: 3


In [5]:
TEXT_MAP = None
TEXT_COL = "comment"

In [6]:
class TransformerDataset(Dataset):
    def __init__(self, frame, tokenizer, max_len, text_col):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.text_col = text_col

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        row = self.frame.iloc[i]
        enc = self.tokenizer(
            str(row[self.text_col]),
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(int(row["label_id"]), dtype=torch.long)
        return item

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    losses, logits_all, y_all = [], [], []

    for batch in loader:
        labels = batch.pop("labels").to(DEVICE)
        inputs = {k: v.to(DEVICE) for k, v in batch.items()}

        logits = model(**inputs).logits
        losses.append(loss_fn(logits, labels).item())
        logits_all.append(logits.cpu())
        y_all.append(labels.cpu())

    logits = torch.cat(logits_all)
    y = torch.cat(y_all).numpy()
    probs = torch.softmax(logits, dim=1).numpy()
    yp = probs.argmax(axis=1)

    m = metrics(y, yp, probs)
    m["loss"] = float(np.mean(losses))
    return m, y, yp, probs

def analyze_token_lengths(tokenizer, texts, hard_max):
    lengths = []
    for i in tqdm(range(0, len(texts), 256), desc="Token-length analysis", leave=False):
        enc = tokenizer(
            texts[i:i+256],
            add_special_tokens=True,
            truncation=False,
            padding=False
        )
        lengths.extend(len(ids) for ids in enc["input_ids"])

    lengths = np.asarray(lengths)
    candidates = [64, 128, 192, 256, 512]
    candidates = [c for c in candidates if c <= hard_max]

    selected = candidates[-1]
    for c in candidates:
        if np.mean(lengths <= c) >= 0.95:
            selected = c
            break

    return {
        "n_samples": int(len(lengths)),
        "p50": float(np.quantile(lengths, 0.50)),
        "p90": float(np.quantile(lengths, 0.90)),
        "p95": float(np.quantile(lengths, 0.95)),
        "p99": float(np.quantile(lengths, 0.99)),
        "coverage_64": float(np.mean(lengths <= 64)),
        "coverage_128": float(np.mean(lengths <= 128)),
        "coverage_192": float(np.mean(lengths <= 192)),
        "coverage_256": float(np.mean(lengths <= 256)),
        "coverage_512": float(np.mean(lengths <= 512)),
        "selected_max_length": int(selected),
    }

def train_task(task, tokenizer, max_len):
    out_dir = ARTIFACT_DIR / MODEL_KEY / task
    pred_path = PRED_DIR / f"{MODEL_KEY}_{task}.csv"
    best_dir = out_dir / "best_model"
    tokenizer_dir = out_dir / "tokenizer"

    if (out_dir / "test_metrics.json").exists() and pred_path.exists():
        print("SKIP completed:", MODEL_KEY, task)
        return

    # Xóa artifact dở dang của lần chạy trước.
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    frames = {s: get_task(s, task) for s in ["train", "dev", "test"]}
    if TEXT_COL != "comment":
        for s in frames:
            frames[s][TEXT_COL] = frames[s]["row_id"].map(TEXT_MAP)
            assert frames[s][TEXT_COL].notna().all()

    w = class_weights(frames["train"]).to(DEVICE)

    def one_attempt(batch_size, grad_accum):
        loaders = {
            s: DataLoader(
                TransformerDataset(frames[s], tokenizer, max_len, TEXT_COL),
                batch_size=batch_size,
                shuffle=(s == "train"),
                num_workers=2,
                pin_memory=True,
            )
            for s in ["train", "dev", "test"]
        }

        seed_all()
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_ID,
            num_labels=2,
            id2label=TASKS[task]["id2label"],
            label2id=TASKS[task]["label2id"],
        ).to(DEVICE)

        loss_fn = nn.CrossEntropyLoss(weight=w)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY
        )

        updates_per_epoch = math.ceil(len(loaders["train"]) / grad_accum)
        total_updates = updates_per_epoch * MAX_EPOCHS
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(total_updates * WARMUP_RATIO),
            num_training_steps=total_updates
        )

        try:
            scaler = torch.amp.GradScaler("cuda", enabled=True)
        except Exception:
            scaler = torch.cuda.amp.GradScaler(enabled=True)

        best_dev_f1 = -1.0
        best_epoch = None
        wait = 0
        history = []
        started = time.time()
        optimizer.zero_grad(set_to_none=True)

        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            train_losses = []

            pbar = tqdm(
                loaders["train"],
                desc=f"{MODEL_KEY} {task} epoch {epoch}"
            )

            for step, batch in enumerate(pbar, start=1):
                labels = batch.pop("labels").to(DEVICE)
                inputs = {k: v.to(DEVICE) for k, v in batch.items()}

                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(**inputs).logits
                    raw_loss = loss_fn(logits, labels)
                    loss = raw_loss / grad_accum

                scaler.scale(loss).backward()
                train_losses.append(raw_loss.item())

                if step % grad_accum == 0 or step == len(loaders["train"]):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)

            dev_m, _, _, _ = evaluate(model, loaders["dev"], loss_fn)

            row = {
                "epoch": epoch,
                "train_loss": float(np.mean(train_losses)),
                "dev_loss": dev_m["loss"],
                "dev_accuracy": dev_m["accuracy"],
                "dev_macro_precision": dev_m["macro_precision"],
                "dev_macro_recall": dev_m["macro_recall"],
                "dev_macro_f1": dev_m["macro_f1"],
                "dev_pr_auc": dev_m["pr_auc_positive_class"],
            }
            history.append(row)
            print(row)

            if dev_m["macro_f1"] > best_dev_f1 + 1e-6:
                best_dev_f1 = dev_m["macro_f1"]
                best_epoch = epoch
                wait = 0

                if best_dir.exists():
                    shutil.rmtree(best_dir)
                if tokenizer_dir.exists():
                    shutil.rmtree(tokenizer_dir)

                model.save_pretrained(best_dir, safe_serialization=True)
                tokenizer.save_pretrained(tokenizer_dir)
            else:
                wait += 1
                if wait >= PATIENCE:
                    print("Early stopping.")
                    break

        training_seconds = time.time() - started
        pd.DataFrame(history).to_csv(
            out_dir / "training_history.csv",
            index=False
        )

        del model
        gc.collect()
        torch.cuda.empty_cache()

        # Restore checkpoint tốt nhất trên DEV trước khi Test.
        model = AutoModelForSequenceClassification.from_pretrained(
            best_dir
        ).to(DEVICE)

        dev_m, _, _, _ = evaluate(model, loaders["dev"], loss_fn)
        test_m, y, yp, probs = evaluate(model, loaders["test"], loss_fn)

        id2label = TASKS[task]["id2label"]
        pred = frames["test"][["row_id", "comment"]].copy()
        pred["gold_label"] = [id2label[int(i)] for i in y]
        pred["predicted_label"] = [id2label[int(i)] for i in yp]
        pred["prob_class_0"] = probs[:, 0]
        pred["prob_class_1"] = probs[:, 1]
        pred["correct"] = pred["gold_label"].eq(pred["predicted_label"])
        pred.to_csv(pred_path, index=False)

        p, r, f, s = precision_recall_fscore_support(
            y, yp, labels=[0, 1], average=None, zero_division=0
        )
        pd.DataFrame([
            {
                "label": id2label[i],
                "precision": p[i],
                "recall": r[i],
                "f1": f[i],
                "support": s[i],
            }
            for i in [0, 1]
        ]).to_csv(out_dir / "classification_report.csv", index=False)

        cm = confusion_matrix(y, yp, labels=[0, 1])
        pd.DataFrame(
            cm,
            index=[id2label[0], id2label[1]],
            columns=[id2label[0], id2label[1]],
        ).to_csv(CM_DIR / f"{MODEL_KEY}_{task}_cm.csv")

        save_json(dev_m, out_dir / "dev_metrics.json")
        save_json(test_m, out_dir / "test_metrics.json")
        save_json({
            "model_key": MODEL_KEY,
            "model_id": MODEL_ID,
            "task": task,
            "seed": SEED,
            "max_epochs": MAX_EPOCHS,
            "early_stopping_patience": PATIENCE,
            "best_epoch": best_epoch,
            "max_length": max_len,
            "physical_batch_size": batch_size,
            "gradient_accumulation_steps": grad_accum,
            "effective_batch_size": batch_size * grad_accum,
            "learning_rate": LR,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "scheduler": "cosine_with_warmup",
            "class_weights": w.detach().cpu().tolist(),
            "training_time_seconds": training_seconds,
            "gpu": torch.cuda.get_device_name(0),
            "torch_version": torch.__version__,
        }, out_dir / "experiment_config.json")

        print("BEST EPOCH:", best_epoch)
        print("TEST:", test_m)

        del model, loaders
        gc.collect()
        torch.cuda.empty_cache()

    # T4-safe, effective batch 32.
    attempts = [(16, 2), (8, 4), (4, 8)]
    last_oom = None

    for batch_size, grad_accum in attempts:
        try:
            one_attempt(batch_size, grad_accum)
            return
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            last_oom = e
            print(f"CUDA OOM với batch={batch_size}; thử batch nhỏ hơn...")
            gc.collect()
            torch.cuda.empty_cache()
            if out_dir.exists():
                shutil.rmtree(out_dir)
            out_dir.mkdir(parents=True, exist_ok=True)

    raise RuntimeError(f"Không train được {MODEL_KEY} {task} do OOM.") from last_oom

In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

hard_max = 512
rows = []

for task in ["task1", "task2"]:
    train_df = get_task("train", task)

    if TEXT_COL != "comment":
        train_df[TEXT_COL] = train_df["row_id"].map(TEXT_MAP)
        assert train_df[TEXT_COL].notna().all()

    stats = analyze_token_lengths(
        tokenizer,
        train_df[TEXT_COL].astype(str).tolist(),
        hard_max
    )
    stats["model"] = MODEL_KEY
    stats["task"] = task
    rows.append(stats)

token_stats = pd.DataFrame(rows)
display(token_stats)

token_file = RESULT_DIR / "token_length_analysis.csv"
if token_file.exists():
    old = pd.read_csv(token_file)
    old = old[old["model"] != MODEL_KEY]
    token_all = pd.concat([old, token_stats], ignore_index=True)
else:
    token_all = token_stats.copy()

token_all.to_csv(token_file, index=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token-length analysis:   0%|          | 0/153 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (571 > 512). Running this sequence through the model will result in indexing errors


Token-length analysis:   0%|          | 0/19 [00:00<?, ?it/s]

,n_samples,p50,p90,p95,p99,coverage_64,coverage_128,coverage_192,coverage_256,coverage_512,selected_max_length,model,task
0,38936,16.0,57.0,84.0,156.00,0.918379,0.981097,0.995326,0.999692,0.999974,128,xlmr,task1
1,4688,24.0,82.0,116.0,191.13,0.846630,0.962671,0.990401,0.999147,1.000000,128,xlmr,task2


In [8]:
# Smoke test trước full fine-tuning.
status = {}
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=2
    ).to(DEVICE)
    status["model_load"] = "PASS"

    sample = get_task("train", "task1").head(2).copy()
    if TEXT_COL != "comment":
        sample[TEXT_COL] = sample["row_id"].map(TEXT_MAP)

    enc = tokenizer(
        sample[TEXT_COL].astype(str).tolist(),
        truncation=True,
        padding=True,
        max_length=64,
        return_tensors="pt"
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    out = model(**enc)
    status["forward"] = "PASS"
    out.logits.sum().backward()
    status["backward"] = "PASS"

    del model, out, enc
    gc.collect()
    torch.cuda.empty_cache()
except Exception as e:
    status["error"] = repr(e)

print(status)
assert status.get("backward") == "PASS"

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'model_load': 'PASS', 'forward': 'PASS', 'backward': 'PASS'}


In [9]:
for task in ["task1", "task2"]:
    max_len = int(
        token_stats[token_stats["task"].eq(task)]["selected_max_length"].iloc[0]
    )
    print("\n" + "=" * 100)
    print(MODEL_KEY.upper(), task.upper(), "max_length =", max_len)
    train_task(task, tokenizer, max_len)


XLMR TASK1 max_length = 128


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


xlmr task1 epoch 1:   0%|          | 0/2434 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.5767952499569883, 'dev_loss': 0.4269265138223523, 'dev_accuracy': 0.8740497226217382, 'dev_macro_precision': 0.717893766258827, 'dev_macro_recall': 0.7862702727266204, 'dev_macro_f1': 0.7441422937714102, 'dev_pr_auc': 0.5919113340983851}


xlmr task1 epoch 2:   0%|          | 0/2434 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 0.4678719211568721, 'dev_loss': 0.3774084505731942, 'dev_accuracy': 0.8666529689747278, 'dev_macro_precision': 0.7146945193025923, 'dev_macro_recall': 0.8166786650753827, 'dev_macro_f1': 0.7482207287186704, 'dev_pr_auc': 0.6650215914810935}


... (34 outputs left out)

xlmr task2 epoch 9:   0%|          | 0/293 [00:00<?, ?it/s]

{'epoch': 9, 'train_loss': 0.19678125388716877, 'dev_loss': 1.1127709896379225, 'dev_accuracy': 0.8191126279863481, 'dev_macro_precision': 0.648937074829932, 'dev_macro_recall': 0.6629302325581395, 'dev_macro_f1': 0.6552558552558553, 'dev_pr_auc': 0.3737235516680098}


xlmr task2 epoch 10:   0%|          | 0/293 [00:00<?, ?it/s]

{'epoch': 10, 'train_loss': 0.20117269781872682, 'dev_loss': 1.1992963647117485, 'dev_accuracy': 0.8327645051194539, 'dev_macro_precision': 0.6575851393188854, 'dev_macro_recall': 0.642046511627907, 'dev_macro_f1': 0.6490160127123823, 'dev_pr_auc': 0.3787383115918692}
BEST EPOCH: 9
TEST: {'accuracy': 0.8378839590443686, 'macro_precision': 0.6864856588751659, 'macro_recall': 0.7124418604651163, 'macro_f1': 0.6977659545590271, 'weighted_f1': 0.8431515531979529, 'pr_auc_positive_class': 0.46279100579791793, 'loss': 1.0352124297437635}


In [10]:
checks = []
for task in ["task1", "task2"]:
    out = ARTIFACT_DIR / MODEL_KEY / task
    checks.append({
        "task": task,
        "best_model": (out / "best_model").exists(),
        "history": (out / "training_history.csv").exists(),
        "dev_metrics": (out / "dev_metrics.json").exists(),
        "test_metrics": (out / "test_metrics.json").exists(),
        "prediction": (PRED_DIR / f"{MODEL_KEY}_{task}.csv").exists(),
        "confusion_matrix": (CM_DIR / f"{MODEL_KEY}_{task}_cm.csv").exists(),
    })

qa = pd.DataFrame(checks)
display(qa)
assert qa.drop(columns=["task"]).all(axis=None)
print(MODEL_KEY.upper(), "COMPLETE")

,task,best_model,history,dev_metrics,test_metrics,prediction,confusion_matrix
0,task1,True,True,True,True,True,True
1,task2,True,True,True,True,True,True


XLMR COMPLETE
